# NSE / BSE Stock Data Fetcher

Fetch free historical OHLCV data — **no account or API key needed**.

| Source | Used for |
|---|---|
| **jugaad-data** | Daily candles — official NSE bhavcopy (most reliable) |
| **yfinance** | Intraday candles — 1m, 5m, 15m, 30m, 1h via Yahoo Finance |

**How to use:**
1. Run **Cell 1** once to install packages.
2. Edit **Cell 3** to set your symbol, dates, and interval.
3. Run **Cell 4** to fetch and display the data.
4. Run **Cell 5** to download the CSV to your device.

In [ ]:
# Cell 1 — Install packages (run once per session)
!pip install -q jugaad-data yfinance pandas

In [ ]:
# Cell 2 — Imports and helper functions
from datetime import date, datetime, timedelta
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from jugaad_data.nse import stock_df as _nse_stock_df
import yfinance as yf

INTRADAY_INTERVALS = {'ONE_MINUTE','TWO_MINUTE','FIVE_MINUTE','FIFTEEN_MINUTE','THIRTY_MINUTE','ONE_HOUR'}
YF_INTERVAL_MAP    = {'ONE_MINUTE':'1m','TWO_MINUTE':'2m','FIVE_MINUTE':'5m',
                      'FIFTEEN_MINUTE':'15m','THIRTY_MINUTE':'30m','ONE_HOUR':'1h',
                      'ONE_DAY':'1d','ONE_WEEK':'1wk','ONE_MONTH':'1mo'}
YF_LOOKBACK_DAYS   = {'1m':7,'2m':60,'5m':60,'15m':60,'30m':60,'1h':730}

def _parse_date(d):
    return d if isinstance(d, date) else datetime.strptime(str(d)[:10], '%Y-%m-%d').date()

def _normalize(df):
    rmap = {'DATE':'DateTime','TIMESTAMP':'DateTime','OPEN':'Open','HIGH':'High',
            'LOW':'Low','CLOSE':'Close','TOTTRDQTY':'Volume','VOLUME':'Volume','TTQ':'Volume'}
    df = df.rename(columns={k:v for k,v in rmap.items() if k in df.columns})
    return df[['DateTime','Open','High','Low','Close','Volume']].copy()

def fetch_daily(symbol, from_date, to_date, series='EQ'):
    from_date, to_date = _parse_date(from_date), _parse_date(to_date)
    print(f'Fetching daily data for {symbol} ({from_date} → {to_date}) via NSE bhavcopy …')
    try:
        raw = _nse_stock_df(symbol=symbol, from_date=from_date, to_date=to_date, series=series)
        if raw is None or raw.empty:
            print('No data returned.'); return None
        df = _normalize(raw)
        df['DateTime'] = pd.to_datetime(df['DateTime'])
        for c in ['Open','High','Low','Close','Volume']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        return df.sort_values('DateTime').reset_index(drop=True)
    except Exception as e:
        print(f'Error: {e}'); return None

def fetch_intraday(symbol, exchange, interval, from_date, to_date):
    from_date, to_date = _parse_date(from_date), _parse_date(to_date)
    yf_iv = YF_INTERVAL_MAP.get(interval)
    if not yf_iv:
        print(f'Unknown interval: {interval}'); return None
    max_days = YF_LOOKBACK_DAYS.get(yf_iv)
    if max_days:
        earliest = date.today() - timedelta(days=max_days)
        if from_date < earliest:
            print(f'Clamping from_date to {earliest} (Yahoo Finance limit for {yf_iv})')
            from_date = earliest
    suffix = '.NS' if exchange.upper() == 'NSE' else '.BO'
    ticker = f'{symbol}{suffix}'
    print(f'Fetching {interval} intraday for {ticker} ({from_date} → {to_date}) via Yahoo Finance …')
    try:
        raw = yf.Ticker(ticker).history(start=str(from_date), end=str(to_date),
                                         interval=yf_iv, auto_adjust=True, prepost=False)
        if raw is None or raw.empty:
            print('No data returned.'); return None
        df = raw[['Open','High','Low','Close','Volume']].copy()
        df.index.name = 'DateTime'
        df = df.reset_index()
        df['DateTime'] = pd.to_datetime(df['DateTime']).dt.tz_localize(None)
        for c in ['Open','High','Low','Close','Volume']:
            df[c] = pd.to_numeric(df[c], errors='coerce')
        return df.sort_values('DateTime').reset_index(drop=True)
    except Exception as e:
        print(f'Error: {e}'); return None

def fetch_historical_data(symbol, from_date, to_date, interval='ONE_DAY', exchange='NSE', series='EQ'):
    if interval in INTRADAY_INTERVALS:
        return fetch_intraday(symbol, exchange, interval, from_date, to_date)
    if exchange.upper() == 'NSE':
        return fetch_daily(symbol, from_date, to_date, series)
    return fetch_intraday(symbol, exchange, interval, from_date, to_date)

print('✓ Functions loaded.')

In [ ]:
# Cell 3 — ✏️  EDIT THIS CELL to configure your fetch

SYMBOL    = 'SBIN'        # NSE ticker, e.g. 'HDFCBANK', 'RELIANCE', 'TCS'
EXCHANGE  = 'NSE'         # 'NSE' or 'BSE'
SERIES    = 'EQ'          # almost always 'EQ' for equities

# Interval options:
#   Daily   : ONE_DAY, ONE_WEEK, ONE_MONTH
#   Intraday: ONE_MINUTE (7d max), FIVE_MINUTE, FIFTEEN_MINUTE, THIRTY_MINUTE, ONE_HOUR (60d max)
INTERVAL  = 'ONE_DAY'

FROM_DATE = '2024-01-01'  # YYYY-MM-DD
TO_DATE   = '2024-12-31'  # YYYY-MM-DD

print(f'Config: {SYMBOL} | {EXCHANGE} | {INTERVAL} | {FROM_DATE} → {TO_DATE}')

In [ ]:
# Cell 4 — Fetch data and display

df = fetch_historical_data(
    symbol=SYMBOL,
    from_date=FROM_DATE,
    to_date=TO_DATE,
    interval=INTERVAL,
    exchange=EXCHANGE,
    series=SERIES,
)

if df is not None:
    print(f'\nRows      : {len(df)}')
    print(f'Date range: {df["DateTime"].min().date()} → {df["DateTime"].max().date()}')
    print(f'Columns   : {list(df.columns)}')
    display(df.head(10))
    display(df.describe().round(2))
else:
    print('Fetch failed — check symbol name and date range.')

In [ ]:
# Cell 5 — Download CSV to your device

if df is not None:
    filename = f'{SYMBOL}_{EXCHANGE}_{INTERVAL}.csv'
    df.to_csv(filename, index=False)

    # In Google Colab, google.colab.files.download() triggers a browser download
    try:
        from google.colab import files
        files.download(filename)
        print(f'Download started: {filename}')
    except ImportError:
        # Running locally (Jupyter) — file is already saved in the current folder
        print(f'Saved to: {filename}')
else:
    print('No data to save — run Cell 4 first.')